# Reaction kinetics: discovering rate laws with BART

Given concentration-vs-time data from a batch reactor, can we recover the
mole-balance right-hand sides (the rate laws)? This is SINDy applied to chemical
kinetics, with **BART** doing the term selection.

The example dataset (`scripts/make_reaction_data.py`) is a liquid-phase batch
reactor with two reactions:

- `A + B -> C`,  rate `r1 = k1 * C_A * C_B`  (bimolecular, k1 = 0.5)
- `C -> D`,      rate `r2 = k2 * C_C`        (first order,  k2 = 0.2)

so the true mole balances are
`dCA/dt = -k1 CA CB`, `dCC/dt = k1 CA CB - k2 CC`, `dCD/dt = k2 CC`.

The measured concentrations carry realistic Gaussian **measurement noise**
(~1%), so the derivatives estimated from them are noisy too -- exactly the
challenge of real kinetic data.

In [ ]:
import numpy as np
from jax_ibart.datasets import load_reaction

# C: concentrations, dCdt: per-batch finite-difference derivatives (the targets)
C, dCdt, species, batch = load_reaction()
print("samples:", C.shape, "| species:", species, "| batches:", len(set(batch)))

The concentration trajectories for one batch (note the measurement scatter):

In [ ]:
import matplotlib.pyplot as plt

m = batch == 0
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for j, s in enumerate(species):
    ax.plot(np.arange(m.sum()), C[m, j], "o", ms=3, alpha=0.7, label=s)
ax.set_xlabel("time index (batch 0)")
ax.set_ylabel("concentration")
ax.set_title("Noisy batch-reactor data")
ax.legend()
fig.tight_layout()
plt.show()

Now let BART find which library terms enter each mole-balance RHS
(`screen="bart"` runs the BART-G.SE nonparametric screen before the sparse l0
fit). Derivatives were estimated per batch, so we pass them via `dXdt`.

In [ ]:
from jax_ibart.dynamics import discover_ode

system = discover_ode(
    C,
    t=None,
    feature_names=species,
    dXdt=dCdt,
    degree=2,
    max_terms=2,
    screen="bart",
    seed=0,
)
print(system)
print(
    "\ntrue: dCA/dt=-0.5 CA CB ; dCB/dt=-0.5 CA CB ; "
    "dCC/dt=0.5 CA CB - 0.2 CC ; dCD/dt=0.2 CC"
)

BART recovers the **bimolecular rate law** `dCA/dt ~ -0.5 C_A C_B` and the
**first-order** `dCD/dt ~ 0.2 C_C` directly from noisy data. The intermediate
`C_C` is the hardest (its derivative is small and noisy, and two terms compete) —
a realistic reminder that intermediates and noisy derivatives are where kinetic
discovery is most delicate; a weak/integral formulation (Messenger & Bortz 2021,
see [references](../references.md)) is the robust next step.

The dimensional-analysis filter (pass `units=...`) would further constrain the
library to rate terms with consistent units, a natural fit for engineering
kinetics.

Rate-fit parity per species (estimated dC/dt vs. the discovered RHS):

In [ ]:
import sympy as sp

syms = sp.symbols("CA CB CC CD")
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for i, (ax, s) in enumerate(zip(axes, species)):
    f = sp.lambdify(syms, system.rhs[i], "numpy")
    pred = f(C[:, 0], C[:, 1], C[:, 2], C[:, 3])
    ax.scatter(dCdt[:, i], np.broadcast_to(pred, dCdt[:, i].shape), s=5, alpha=0.3)
    lo, hi = dCdt[:, i].min(), dCdt[:, i].max()
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_title(f"d{s}/dt (R$^2$={system.r2[i]:.2f})")
    ax.set_xlabel("measured")
    ax.set_ylabel("fitted")
fig.tight_layout()
plt.show()

The bimolecular species (`A`, `B`) and the product `C` fit tightly; the weak,
slow `dCD/dt` is noise-dominated (small derivative), so its parity scatters --
the honest signature of a hard-to-identify slow step.